# Eigenvalues and Eigenvectors

| Difficulty | ████░░░░░░ 4/10 |
| :--- | :--- |
| Prerequisites | 01.02, 01.05 |
| Time | ~70 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/01_Linear_Algebra/06_eigenvalues_eigenvectors.ipynb)

---

## 🎯 Learning Objective

Learn what eigenvalues and eigenvectors reveal about a linear transformation's natural axes, and understand positive-definiteness and condition number as diagnostics for optimization.

---

## 📐 Theory

An **eigenvector** of a square matrix $A$ is a nonzero vector $\mathbf{v}$ whose direction
$A$ does not change — $A$ only stretches or shrinks it by a scalar factor $\lambda$, its
**eigenvalue**:

$$A\mathbf{v} = \lambda\mathbf{v}$$

Every other vector gets rotated *and* scaled by $A$; eigenvectors are the special directions
that only get scaled. Finding them means solving $\det(A - \lambda I) = 0$ — the
**characteristic equation**.

### Eigendecomposition and the spectral theorem

If $A$ (size $n\times n$) has $n$ linearly independent eigenvectors, it can be written as

$$A = V\Lambda V^{-1}$$

where columns of $V$ are the eigenvectors and $\Lambda$ is diagonal with the eigenvalues. This
**eigendecomposition** re-expresses $A$'s action as "change basis to the eigenvectors, scale
each axis independently, change back."

The **spectral theorem** (for the real symmetric case; proof in Strang, *Introduction to Linear Algebra*, Ch. 6, cited in Further Reading) gives a much nicer guarantee for **symmetric** matrices ($A=A^\top$,
which covariance matrices always are): their eigenvectors can always be chosen orthogonal, so
$A = Q\Lambda Q^\top$ with $Q$ orthogonal ($Q^\top Q = I$) — decomposition without ever needing
a numerically fragile inverse.

### Positive definiteness

A symmetric matrix $A$ is **positive definite** if $\mathbf{x}^\top A\mathbf{x} > 0$ for every
nonzero $\mathbf{x}$ — equivalently, every eigenvalue of $A$ is strictly positive.
Positive-*semi*-definite allows eigenvalues of exactly zero. This matters constantly: the
Hessian of a loss function being positive definite at a point is exactly the condition for
that point to be a local minimum (not a saddle point).

### Condition number

$$\kappa(A) = \frac{\lambda_{\max}}{\lambda_{\min}}$$

(ratio of largest to smallest eigenvalue/singular value magnitude). A large condition number
means $A$ stretches some directions vastly more than others — exactly the "nearly singular"
sensitivity we saw break numerical stability in
[Notebook 01.05](05_determinants_and_inverses.ipynb).

---

## 👁️ Visual Intuition

Eigenvectors are the only directions a transformation doesn't rotate. Overlaying eigenvector
arrows on a transformed grid of points shows them staying put in direction while everything
else swings around.

In [ ]:
# Setup + eigenvectors as the "unrotated" directions of a linear map
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

A = np.array([[3.0, 1.0], [0.0, 2.0]])
eigvals, eigvecs = np.linalg.eig(A)

theta = np.linspace(0, 2 * np.pi, 60)
circle = np.stack([np.cos(theta), np.sin(theta)])
transformed_circle = A @ circle

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(*circle, color="lightgray", label="unit circle")
ax.plot(*transformed_circle, color="#4C72B0", label="A @ unit circle")
for i in range(2):
    v = eigvecs[:, i]
    lam = eigvals[i]
    ax.quiver(0, 0, v[0], v[1], color="green", angles='xy', scale_units='xy', scale=1,
              width=0.01, label=f"eigenvector {i+1} ($\\lambda$={lam:.1f})")
    ax.quiver(0, 0, lam * v[0], lam * v[1], color="red", angles='xy', scale_units='xy', scale=1,
              width=0.008, alpha=0.6)
ax.set_xlim(-4, 4); ax.set_ylim(-4, 4); ax.set_aspect('equal')
ax.set_title("Eigenvectors (green) keep their direction; red shows A applied to them")
ax.legend(fontsize=8)
plt.show()

## 🐍 Implementation from Scratch

The power iteration method finds the dominant eigenvector using nothing but repeated
matrix-vector multiplication — a good way to *feel* what "the direction A stretches most"
means, and the same core idea that scales up to huge matrices where full eigendecomposition
would be too expensive.

In [ ]:
# Power iteration: find the dominant eigenvector/eigenvalue via repeated multiplication
import numpy as np

def power_iteration(A, n_iter=100):
    n = A.shape[0]
    v = np.random.default_rng(0).standard_normal(n)
    v = v / np.linalg.norm(v)
    for _ in range(n_iter):
        v = A @ v
        v = v / np.linalg.norm(v)             # renormalize each step to avoid overflow/underflow
    eigenvalue = v @ A @ v                     # Rayleigh quotient (v is now a unit eigenvector)
    return eigenvalue, v

A = np.array([[4.0, 1.0], [2.0, 3.0]])
lam_power, v_power = power_iteration(A)

eigvals_np, eigvecs_np = np.linalg.eig(A)
dominant_idx = np.argmax(np.abs(eigvals_np))

print("Power iteration eigenvalue :", round(lam_power, 4))
print("np.linalg.eig dominant     :", round(eigvals_np[dominant_idx].real, 4))
print("Power iteration eigenvector:", np.round(v_power, 4))
print("np.linalg.eig eigenvector  :", np.round(eigvecs_np[:, dominant_idx].real, 4), "(up to sign)")

# Positive-definiteness check via eigenvalues
def is_positive_definite(A):
    eigvals = np.linalg.eigvalsh(A)   # eigvalsh assumes/exploits symmetry, returns REAL eigenvalues
    return np.all(eigvals > 0)

cov_like = np.array([[2.0, 0.5], [0.5, 1.0]])
print("\nIs a covariance-like matrix positive definite?", is_positive_definite(cov_like))
print("Condition number (lambda_max / lambda_min):", round(np.linalg.cond(cov_like), 4))

## 🤖 Why This Matters for AI

**PCA is an eigendecomposition** of the data covariance matrix: the eigenvectors are the
principal directions of variance, and eigenvalues tell you how much variance each direction
explains. Below we run PCA "by hand" via eigendecomposition and confirm it matches
scikit-learn, previewing [Notebook 01.07](07_SVD_and_PCA.ipynb) which does the same thing via
SVD (the numerically preferred route in practice).

**Condition number and training stability**: the condition number of the loss Hessian governs
how fast (or slowly, or unstably) gradient descent converges — a running theme picked up again
in Module 04 (Optimization).

In [ ]:
# PCA "by hand" via eigendecomposition of the covariance matrix
import numpy as np
from sklearn.decomposition import PCA

rng = np.random.default_rng(3)
# Correlated 2D data: variance is much larger along one direction than the other
mean = np.array([0.0, 0.0])
cov_true = np.array([[3.0, 1.5], [1.5, 1.0]])
X = rng.multivariate_normal(mean, cov_true, size=500)

X_centered = X - X.mean(axis=0)
cov_empirical = (X_centered.T @ X_centered) / (X.shape[0] - 1)

eigvals, eigvecs = np.linalg.eigh(cov_empirical)     # eigh: symmetric-matrix eigendecomposition
order = np.argsort(eigvals)[::-1]                     # sort descending: largest-variance direction first
eigvals, eigvecs = eigvals[order], eigvecs[:, order]

print("Eigenvalues (variance explained by each principal component):", np.round(eigvals, 3))
print("Fraction of variance explained by PC1:", round(eigvals[0] / eigvals.sum(), 4))

pca = PCA(n_components=2).fit(X)
print("\nsklearn PCA explained_variance_:      ", np.round(pca.explained_variance_, 3))
print("Our eigendecomposition matches sklearn:", np.allclose(sorted(eigvals), sorted(pca.explained_variance_), atol=1e-1))
print("\n(small numerical differences are expected -- sklearn PCA uses SVD internally, see 01.07)")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. By hand, find the eigenvalues and eigenvectors of the notebook's own power-iteration matrix
   $A=\begin{pmatrix}4&1\\2&3\end{pmatrix}$ by solving $\det(A-\lambda I)=0$, then verify with
   `np.linalg.eig`. As a sanity check, confirm $A\mathbf{v}=\lambda\mathbf{v}$ holds exactly for
   each eigenpair you found.

<details>
<summary>💡 Solution</summary>

$\det(A-\lambda I)=(4-\lambda)(3-\lambda)-2=\lambda^2-7\lambda+10=(\lambda-5)(\lambda-2)=0$, so
$\lambda_1=5,\lambda_2=2$ — matching `np.linalg.eig(A)` exactly. For $\lambda_1=5$:
$(A-5I)=\begin{pmatrix}-1&1\\2&-2\end{pmatrix}$, whose rows are proportional (rank-deficient as
required), giving eigenvector direction $\mathbf{v}_1=(1,1)$. For $\lambda_2=2$:
$(A-2I)=\begin{pmatrix}2&1\\2&1\end{pmatrix}$, giving $\mathbf{v}_2=(1,-2)$. Check:
$A\mathbf{v}_1=(5,5)=5\mathbf{v}_1$ ✓ and $A\mathbf{v}_2=(2,-4)=2\mathbf{v}_2$ ✓, both exact.

</details>

### 💭 UNDERSTAND
2. The Theory section notes covariance matrices are symmetric, and the spectral theorem
   guarantees symmetric matrices have real eigenvalues with orthogonal eigenvectors. The
   rotation matrix $R(90°)=\begin{pmatrix}0&-1\\1&0\end{pmatrix}$ is *not* symmetric. Without
   computing anything yet, do you expect it to have real eigenvalues? (Hint: think about what a
   real eigenvector of a pure rotation would have to mean geometrically — a direction rotation
   leaves unchanged.) Compute its eigenvalues to check, and connect the result back to the
   geometric picture.

<details>
<summary>💡 Solution</summary>

$R(90°)$ has no real eigenvalues at all — `np.linalg.eig` returns $\lambda=\pm i$, a complex
conjugate pair, with det = 1 = |λ₁||λ₂| and no real solution to $\det(R-\lambda I)=0$. This
matches the geometric intuition directly: a real eigenvector would have to be a direction that
$R(90°)$ maps to a scalar multiple of itself, but a $90°$ rotation turns *every* nonzero vector
in the plane to point in a genuinely different direction — there is no such direction, so there
can be no real eigenvector, and the algebra (complex eigenvalues) is just confirming what the
geometry already rules out. Symmetric matrices are exactly the ones guaranteed to avoid this:
the spectral theorem's real-eigenvalue guarantee is a real restriction on which matrices qualify,
not a universal fact about all square matrices.

</details>

### ✏️ DERIVE
3. Prove that a symmetric matrix's eigenvectors corresponding to *distinct* eigenvalues are
   orthogonal: for symmetric $A$ with $A\mathbf{v}_1=\lambda_1\mathbf{v}_1$,
   $A\mathbf{v}_2=\lambda_2\mathbf{v}_2$, and $\lambda_1\ne\lambda_2$, show
   $\mathbf{v}_1\cdot\mathbf{v}_2=0$.

<details>
<summary>💡 Solution outline</summary>

Consider $\mathbf{v}_1^\top A\mathbf{v}_2$, computed two ways. First, using
$A\mathbf{v}_2=\lambda_2\mathbf{v}_2$ directly:
$$\mathbf{v}_1^\top A\mathbf{v}_2 = \lambda_2\,\mathbf{v}_1^\top\mathbf{v}_2.$$
Second, use symmetry ($A^\top=A$) to rewrite $\mathbf{v}_1^\top A\mathbf{v}_2 =
(A^\top\mathbf{v}_1)^\top\mathbf{v}_2 = (A\mathbf{v}_1)^\top\mathbf{v}_2 =
(\lambda_1\mathbf{v}_1)^\top\mathbf{v}_2 = \lambda_1\,\mathbf{v}_1^\top\mathbf{v}_2$. The two
expressions must be equal (they're the same quantity, $\mathbf{v}_1^\top A\mathbf{v}_2$,
computed two ways):
$$\lambda_1\,\mathbf{v}_1^\top\mathbf{v}_2 = \lambda_2\,\mathbf{v}_1^\top\mathbf{v}_2
\;\;\Longrightarrow\;\; (\lambda_1-\lambda_2)\,\mathbf{v}_1^\top\mathbf{v}_2 = 0.$$
Since $\lambda_1\ne\lambda_2$ by assumption, the factor $(\lambda_1-\lambda_2)$ is nonzero, so
$\mathbf{v}_1^\top\mathbf{v}_2$ must be exactly zero — i.e. $\mathbf{v}_1\perp\mathbf{v}_2$.
Numeric check on the notebook's own `cov_like`-style symmetric matrix
$S=\begin{pmatrix}2&0.5\\0.5&1\end{pmatrix}$: eigenvalues $\approx2.207,0.793$ (distinct), and
their eigenvectors' dot product is exactly $0$ to floating-point precision — confirmed again on
a random $4\times4$ symmetric matrix with 4 distinct eigenvalues, where the full eigenvector
matrix $V$ satisfies $V^\top V=I$ (every pair orthogonal, all unit norm), not just one pair.

</details>

### 🐍 IMPLEMENT
4. `power_iteration` finds only the dominant (largest-magnitude) eigenvalue. Implement
   **deflation** to recover the *second* eigenvalue too: after running `power_iteration` to get
   $(\lambda_1,\mathbf{v}_1)$ (with $\mathbf{v}_1$ unit norm), form
   $A' = A - \lambda_1\mathbf{v}_1\mathbf{v}_1^\top$ and run `power_iteration` again on $A'$.
   Test on a random symmetric $4\times4$ matrix.

<details>
<summary>✅ How to know you're right</summary>

Both recovered eigenvalues must match the two largest-magnitude entries of
`np.linalg.eigh(A)`'s output, in the correct order — for a sample random symmetric $4\times4$
matrix with true eigenvalues (sorted by magnitude) $\approx4.70,-2.79,1.40,0.35$, deflation
should recover $\approx4.70$ then $\approx-2.79$, matching to the same tolerance
`power_iteration` itself achieves on the dominant eigenvalue alone. The mechanism to check if it
doesn't work: deflation only works cleanly if $\mathbf{v}_1$ is truly unit norm (renormalize it
explicitly before building the outer product) — a $\mathbf{v}_1$ that's off by even a small
scale factor will corrupt $A'$'s remaining eigenvalues, not just the one you're trying to remove.

</details>

### 🤖 APPLY
5. A loss function's Hessian at a critical point is $H=\begin{pmatrix}3&1\\1&3\end{pmatrix}$ for
   one candidate point and $H=\begin{pmatrix}1&2\\2&1\end{pmatrix}$ for another. Using
   eigenvalues (not the raw matrix entries), classify each critical point as a local minimum or
   a saddle point, and for the saddle, find the specific direction $\mathbf{x}$ along which the
   loss actually *decreases* despite the point being a critical point (zero gradient).

<details>
<summary>✅ What you should observe</summary>

The first Hessian has eigenvalues $2$ and $4$ — both strictly positive, so
$\mathbf{x}^\top H\mathbf{x}>0$ for every direction (verified directly on several random test
vectors, all giving a positive value) — a genuine local minimum. The second has eigenvalues
$-1$ and $3$ — one negative, one positive, so this point is a saddle, not a minimum. The
decreasing direction is exactly the eigenvector for the negative eigenvalue, $\mathbf{v}\approx
(-0.707,0.707)$: moving along $\mathbf{v}$ gives $\mathbf{v}^\top H\mathbf{v}=-1$ exactly (the
eigenvalue itself, by the defining property $H\mathbf{v}=\lambda\mathbf{v}$), confirming the
loss curves *downward* in that specific direction even at a point with zero gradient. This is
exactly why second-order optimization diagnostics (and saddle-point-escaping methods) look at
the Hessian's eigenvalues rather than just checking "is the gradient zero" — zero gradient alone
can't distinguish a minimum from a saddle, but the sign pattern of the Hessian's eigenvalues
always can.

</details>

### 🚀 CHALLENGE
6. Generate a positive-definite $3\times3$ matrix as $A=B^\top B$ for random $B$ (explain in a
   comment why this construction always guarantees positive-*semi*-definiteness, and what
   extra condition on $B$ upgrades that to strictly positive-definite). Compute its condition
   number, then construct a second, deliberately ill-conditioned symmetric matrix with the same
   eigenvectors but a condition number orders of magnitude larger (e.g. eigenvalues
   $100, 1, 10^{-4}$ instead of $A$'s own). Solve $M\mathbf{x}=\mathbf{b}$ for both matrices, add
   a tiny perturbation to $\mathbf{b}$, and compare how much each solution moves.

<details>
<summary>🔍 What a strong answer covers</summary>

$B^\top B$ is positive-semi-definite for *any* $B$ because $\mathbf{x}^\top(B^\top B)\mathbf{x} =
(B\mathbf{x})^\top(B\mathbf{x}) = \|B\mathbf{x}\|_2^2 \ge 0$ for every $\mathbf{x}$ — a squared
norm can never be negative. It's strictly positive-definite specifically when $B$ has full
column rank (no nonzero $\mathbf{x}$ satisfies $B\mathbf{x}=\mathbf{0}$), since that's exactly
when $\|B\mathbf{x}\|_2^2>0$ for every nonzero $\mathbf{x}$. A strong answer confirms this
numerically (a random $3\times3$ $B$ gives $A=B^\top B$ with eigenvalues like
$\approx0.093,1.998,3.703$ — all strictly positive, $\text{cond}(A)\approx39.9$) and then builds
the ill-conditioned contrast case with eigenvalues $100,1,10^{-4}$ on the *same* eigenvectors
(via $Q\,\text{diag}(100,1,10^{-4})\,Q^\top$ for a random orthogonal $Q$), giving
$\text{cond}\approx10^6$. Perturbing $\mathbf{b}$ by a relative amount of order $10^{-6}$
produces a relative change in $\mathbf{x}$ of about $8\times10^{-2}$ for the well-conditioned
$A$ (amplification factor $\approx0.08$, i.e. *damped*, not amplified) versus about $4.2$ for the
ill-conditioned matrix (amplification factor $>1$, i.e. genuinely amplified) — a difference of
roughly $50\times$ between the two amplification factors, tracking the two condition numbers'
own ratio reasonably well. A strong answer connects the specific mechanism: the tiny eigenvalue
($10^{-4}$) means that direction in the solution is divided by a very small number when solving
$M\mathbf{x}=\mathbf{b}$, so whatever small perturbation happens to land along that eigenvector
gets massively amplified — exactly the mechanism condition number quantifies, tying together
`01.05`'s determinant-based numerical instability story and the Hessian-conditioning story that
Module 04 (Optimization) picks up again for gradient descent's convergence rate.

</details>

---

## 📚 Further Reading
- Strang, *Introduction to Linear Algebra*, Ch. 6 (Eigenvalues and Eigenvectors)
- [3Blue1Brown: Eigenvectors and eigenvalues](https://www.3blue1brown.com/lessons/eigenvalues)

---

*Next notebook: [SVD and PCA](07_SVD_and_PCA.ipynb)*